In [1]:
import os
import sys
sys.path.append(os.path.abspath('..'))

from dotenv import load_dotenv
from pathlib import Path
from utils.utils import sliding_windows
import joblib
import numpy as np
from scipy.stats import zscore
from scipy.signal import butter, sosfiltfilt

load_dotenv('../.env')

True

In [2]:
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

BASE_PATH = os.getenv("BASE_PATH")
PICKLE_PATH = BASE_PATH + os.getenv("PICKLE_PATH")

In [3]:
WINDOWS_SETUP = [
    [1, 0.5],
    [1, 1],
    [1.5, 0.5],
    [1.5, 1],
    [1.5, 1.5],
    [2, 0.5],
    [2, 1],
    [2, 1.5],
    [2, 2]
]

SEEDS = [0, 1, 42, 123, 2024]

eo_data = joblib.load(PICKLE_PATH + 'eo_crop.pkl')
ec_data = joblib.load(PICKLE_PATH + 'ec_crop.pkl')


def split_raw_by_time(raw_list, train_ratio=0.9):
    """
    Split raw EEG data by TIME for each subject.
    All subjects appear in both train and test sets (different time segments).
    This is correct for IDENTIFICATION tasks.
    """
    train_raw, test_raw = [], []

    for raw in raw_list:
        sfreq = raw.info['sfreq']
        n_times = raw.n_times
        train_end = int(n_times * train_ratio)

        train_raw.append(raw.copy().crop(tmin=0, tmax=(train_end - 1) / sfreq))
        test_raw.append(raw.copy().crop(tmin=train_end / sfreq, tmax=(n_times - 1) / sfreq))

    return train_raw, test_raw


def butter_bandpass_filter(raw, lowcut=4, highcut=40, fs=160, order=5):
    """Apply Butterworth bandpass filter to a single Raw object, returns filtered numpy array (channels, time)."""
    nyq = 0.5 * fs
    sos = butter(order, [lowcut / nyq, highcut / nyq], btype='band', output='sos')
    data = []
    for subject in raw:
        data.append(subject.get_data())
    return sosfiltfilt(sos, np.stack(data), axis=1)


# Time-based split (same subjects in train and test, different time segments)
eo_train_raw, eo_test_raw = split_raw_by_time(eo_data)
ec_train_raw, ec_test_raw = split_raw_by_time(ec_data)

print(f"Subjects in train: {len(eo_train_raw)}, Subjects in test: {len(eo_test_raw)}")

Subjects in train: 109, Subjects in test: 109


In [4]:
def process_and_save_data(raw_train, raw_test, prefix, window_size, stride, seed, sfreq, output_dir):
    """Process EEG data: windowing, filtering, normalization, and save."""        
    # Butterworth bandpass filter (4-40 Hz, order 5)
    filtered_train_data = butter_bandpass_filter(raw_train, lowcut=4, highcut=40, fs=sfreq, order=5)
    filtered_test_data = butter_bandpass_filter(raw_test, lowcut=4, highcut=40, fs=sfreq, order=5)

    # Sliding windows
    X_train, y_train = sliding_windows(filtered_train_data, window_size, stride, sfreq)
    X_test, y_test = sliding_windows(filtered_test_data, window_size, stride, sfreq)

    # Convert to numpy arrays
    X_train, X_test = np.array(X_train), np.array(X_test)
    y_train, y_test = np.array(y_train), np.array(y_test)
    
    print(f"  {prefix} train/test: {X_train.shape}, {X_test.shape}")
    print(f"  Train subjects: {len(np.unique(y_train))}, Test subjects: {len(np.unique(y_test))}")
    
    # Z-score normalization
    X_train = zscore(X_train, axis=2)
    X_test = zscore(X_test, axis=2)
    
    # Save to disk (seed in filename for compatibility, but split is time-based)
    suffix = f"{str(window_size).replace('.', '')}_{str(stride).replace('.', '')}_seed{seed}"
    for name, data in [('X_train', X_train), ('X_test', X_test),
                       ('y_train', y_train), ('y_test', y_test)]:
        np.save(output_dir / f'{name.replace("_", f"_{prefix.lower()}_", 1)}_{suffix}.npy', data)
    
    return X_train, y_train

# Setup
sfreq = eo_data[0].info['sfreq']
print(f"Sampling frequency: {sfreq} Hz")

PREPROCESSED_PATH = BASE_PATH + os.getenv("PREPROCESSED_PATH")
PREPROCESSED_DIR = Path(PREPROCESSED_PATH)
print(f"Preprocessed data will be saved to: {PREPROCESSED_DIR}")
PREPROCESSED_DIR.mkdir(exist_ok=True)

# Time-based split is deterministic, so we only need to process once per window config
# But we save with different seed names for compatibility with training script
for seed in SEEDS:
    print(f"\n{'='*50}")
    print(f"SEED: {seed} (filename only - split is time-based)")
    print(f"{'='*50}")
    
    for window_size, stride in WINDOWS_SETUP:
        print(f"\n--- Window: {window_size}s, Stride: {stride}s ---")
        
        X_eo_train, y_eo_train = process_and_save_data(
            eo_train_raw, eo_test_raw, "EO", window_size, stride, seed, sfreq, PREPROCESSED_DIR
        )
        X_ec_train, y_ec_train = process_and_save_data(
            ec_train_raw, ec_test_raw, "EC", window_size, stride, seed, sfreq, PREPROCESSED_DIR
        )

Sampling frequency: 160.0 Hz
Preprocessed data will be saved to: Dataset/preprocessed_research_final_v4_90

SEED: 0 (filename only - split is time-based)

--- Window: 1s, Stride: 0.5s ---
  EO train/test: (11663, 64, 160), (1199, 64, 160)
  Train subjects: 109, Test subjects: 109
  EC train/test: (11663, 64, 160), (1199, 64, 160)
  Train subjects: 109, Test subjects: 109

--- Window: 1s, Stride: 1s ---
  EO train/test: (5886, 64, 160), (654, 64, 160)
  Train subjects: 109, Test subjects: 109
  EC train/test: (5886, 64, 160), (654, 64, 160)
  Train subjects: 109, Test subjects: 109

--- Window: 1.5s, Stride: 0.5s ---
  EO train/test: (11554, 64, 240), (1090, 64, 240)
  Train subjects: 109, Test subjects: 109
  EC train/test: (11554, 64, 240), (1090, 64, 240)
  Train subjects: 109, Test subjects: 109

--- Window: 1.5s, Stride: 1s ---
  EO train/test: (5777, 64, 240), (545, 64, 240)
  Train subjects: 109, Test subjects: 109
  EC train/test: (5777, 64, 240), (545, 64, 240)
  Train subjects

In [5]:
print("Proses Selesai! Data siap masuk Embedding Model.")
print("Verifikasi Mean (harus ~0):", np.mean(X_eo_train[0, 0, :]))
print("Verifikasi Std (harus 1):", np.std(X_eo_train[0, 0, :]))

Proses Selesai! Data siap masuk Embedding Model.
Verifikasi Mean (harus ~0): -2.2204460492503132e-17
Verifikasi Std (harus 1): 0.9999999999999998
